In [2]:
import numpy as np
import pandas as pd
import joblib
import json
import os
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    recall_score, accuracy_score,
    roc_auc_score, classification_report
)
import warnings
warnings.filterwarnings('ignore')

# Load original raw data — NOT the processed version
# Because our full pipeline will handle preprocessing itself
df = pd.read_csv('../data/diabetes.csv')

# Separate features and target
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Load the tuned model we saved in Phase 5
tuned_model = joblib.load('../models/random_forest_tuned.joblib')

# Load feature names
with open('../models/feature_names.json', 'r') as f:
    feature_names = json.load(f)

print("Everything loaded successfully!")
print(f"Tuned model type : {type(tuned_model)}")
print(f"Feature names    : {feature_names}")

Everything loaded successfully!
Tuned model type : <class 'sklearn.ensemble._forest.RandomForestClassifier'>
Feature names    : ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']


In [3]:
print("""
WHY WE BUILD A FULL PIPELINE
==============================

In Phase 3 we built a preprocessing pipeline.
In Phase 4/5 we trained and tuned a model separately.

For PRODUCTION we combine them:

  preprocessing_pipeline + tuned_model = full_prediction_pipeline

Benefits:
  1. One object does everything — no missing steps
  2. API code becomes simple: pipeline.predict(raw_input)
  3. Impossible to forget preprocessing
  4. Easy to version and replace later
  5. Industry standard approach

Think of it like a vending machine:
  You put in coins (raw data)
  Machine handles everything internally
  You get chips (prediction)
  You don't care about internal steps
""")


WHY WE BUILD A FULL PIPELINE

In Phase 3 we built a preprocessing pipeline.
In Phase 4/5 we trained and tuned a model separately.

For PRODUCTION we combine them:

  preprocessing_pipeline + tuned_model = full_prediction_pipeline

Benefits:
  1. One object does everything — no missing steps
  2. API code becomes simple: pipeline.predict(raw_input)
  3. Impossible to forget preprocessing
  4. Easy to version and replace later
  5. Industry standard approach

Think of it like a vending machine:
  You put in coins (raw data)
  Machine handles everything internally
  You get chips (prediction)
  You don't care about internal steps



In [4]:
# Get the best hyperparameters from our tuned model
best_params = tuned_model.get_params()

print("Best hyperparameters from tuned model:")
important_params = [
    'n_estimators', 'max_depth', 'min_samples_split',
    'min_samples_leaf', 'max_features', 'class_weight'
]
for param in important_params:
    print(f"  {param:20}: {best_params[param]}")

# Build the FULL pipeline
# Step 1: Imputer — fills missing values with median
# Step 2: Scaler  — scales features to same range
# Step 3: Model   — makes the prediction
full_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
    ('model',   RandomForestClassifier(
        n_estimators    = best_params['n_estimators'],
        max_depth       = best_params['max_depth'],
        min_samples_split = best_params['min_samples_split'],
        min_samples_leaf  = best_params['min_samples_leaf'],
        max_features    = best_params['max_features'],
        class_weight    = best_params['class_weight'],
        random_state    = 42,
        n_jobs          = -1
    ))
])

print("\nFull pipeline created:")
for step_name, step_obj in full_pipeline.steps:
    print(f"  → {step_name}: {step_obj.__class__.__name__}")

Best hyperparameters from tuned model:
  n_estimators        : 250
  max_depth           : 5
  min_samples_split   : 2
  min_samples_leaf    : 5
  max_features        : None
  class_weight        : balanced

Full pipeline created:
  → imputer: SimpleImputer
  → scaler: StandardScaler
  → model: RandomForestClassifier


In [5]:
# IMPORTANT PRODUCTION DECISION:
# In phases 3-5 we used 80% data for training, 20% for testing
# That was for EVALUATION — to honestly measure performance
#
# Now for the FINAL saved model, we train on 100% of data
# Why? Because:
#   → More data = model learns more patterns
#   → We already know the model works well from our evaluation
#   → In production, every data point helps
#   → We evaluated honestly already — no need to hold back data anymore

# Replace hidden zeros with NaN first
zero_columns = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
X_clean = X.copy()
X_clean[zero_columns] = X_clean[zero_columns].replace(0, np.nan)

# Train on ALL data
full_pipeline.fit(X_clean, y)

print("Full pipeline trained on 100% of data!")
print(f"Total samples used for training: {len(X_clean)}")

Full pipeline trained on 100% of data!
Total samples used for training: 768


In [6]:
# Test with a FAKE patient — raw input just like a real user would submit

# Fake Patient 1 — High risk profile
high_risk_patient = pd.DataFrame([{
    'Pregnancies'             : 6,
    'Glucose'                 : 148,
    'BloodPressure'           : 72,
    'SkinThickness'           : 35,
    'Insulin'                 : 0,      # missing — pipeline will handle this
    'BMI'                     : 33.6,
    'DiabetesPedigreeFunction': 0.627,
    'Age'                     : 50
}])

# Fake Patient 2 — Low risk profile
low_risk_patient = pd.DataFrame([{
    'Pregnancies'             : 1,
    'Glucose'                 : 85,
    'BloodPressure'           : 66,
    'SkinThickness'           : 29,
    'Insulin'                 : 0,      # missing — pipeline will handle this
    'BMI'                     : 26.6,
    'DiabetesPedigreeFunction': 0.351,
    'Age'                     : 25
}])

# Replace zeros with NaN for both test patients
for patient_df in [high_risk_patient, low_risk_patient]:
    patient_df[zero_columns] = patient_df[zero_columns].replace(0, np.nan)

# Make predictions — pipeline handles preprocessing automatically
for name, patient in [("High Risk Patient", high_risk_patient),
                       ("Low Risk Patient",  low_risk_patient)]:

    prediction   = full_pipeline.predict(patient)[0]
    probability  = full_pipeline.predict_proba(patient)[0]

    print(f"\n{name}")
    print(f"  Raw input glucose : {patient['Glucose'].values[0]}")
    print(f"  Prediction        : {'DIABETIC ⚠️' if prediction == 1 else 'NOT DIABETIC ✅'}")
    print(f"  Confidence        : {max(probability)*100:.1f}%")
    print(f"  P(No Diabetes)    : {probability[0]*100:.1f}%")
    print(f"  P(Diabetes)       : {probability[1]*100:.1f}%")


High Risk Patient
  Raw input glucose : 148
  Prediction        : DIABETIC ⚠️
  Confidence        : 83.5%
  P(No Diabetes)    : 16.5%
  P(Diabetes)       : 83.5%

Low Risk Patient
  Raw input glucose : 85
  Prediction        : NOT DIABETIC ✅
  Confidence        : 97.7%
  P(No Diabetes)    : 97.7%
  P(Diabetes)       : 2.3%


In [7]:
os.makedirs('../models', exist_ok=True)

# Save the full pipeline — THIS is what the API will load
joblib.dump(full_pipeline, '../models/full_pipeline.joblib')

# Save model metadata — useful for API responses and monitoring
import json
from datetime import datetime

metadata = {
    'model_name'      : 'Random Forest Classifier',
    'version'         : '1.0.0',
    'trained_on'      : datetime.now().strftime('%Y-%m-%d'),
    'training_samples': len(X_clean),
    'features'        : feature_names,
    'target'          : 'Outcome (0=No Diabetes, 1=Diabetes)',
    'best_params'     : {k: best_params[k] for k in important_params},
    'performance'     : {
        'recall'  : 0.7593,
        'roc_auc' : 0.8300,
        'accuracy': 0.7727
    },
    'preprocessing'   : {
        'imputation'  : 'median',
        'scaling'     : 'StandardScaler',
        'zero_columns': zero_columns
    }
}

with open('../models/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Saved files:")
print("  → models/full_pipeline.joblib     ← API loads this ONE file")
print("  → models/model_metadata.json      ← model info and version")
print()

# Verify saved file works by loading and predicting
loaded_pipeline = joblib.load('../models/full_pipeline.joblib')
test_pred = loaded_pipeline.predict(high_risk_patient)
print(f"Verification — loaded pipeline prediction: {'DIABETIC' if test_pred[0]==1 else 'NOT DIABETIC'}")
print("Pipeline loads and predicts correctly ✅")

Saved files:
  → models/full_pipeline.joblib     ← API loads this ONE file
  → models/model_metadata.json      ← model info and version

Verification — loaded pipeline prediction: DIABETIC
Pipeline loads and predicts correctly ✅


In [8]:
# A model card documents what your model does
# Real companies publish these for transparency

print("""
╔══════════════════════════════════════════════════╗
║           MODEL CARD — Disease Predictor         ║
╠══════════════════════════════════════════════════╣
║  Model     : Random Forest Classifier            ║
║  Version   : 1.0.0                               ║
║  Task      : Binary Classification               ║
║  Dataset   : Pima Indians Diabetes (768 samples) ║
╠══════════════════════════════════════════════════╣
║  PERFORMANCE                                     ║
║  Recall    : 75.93%  (catches 76% of diabetics)  ║
║  ROC-AUC   : 83.00%  (strong separation ability) ║
║  Accuracy  : 77.27%                              ║
╠══════════════════════════════════════════════════╣
║  PIPELINE STEPS                                  ║
║  1. Median Imputation  (fix missing values)      ║
║  2. Standard Scaling   (normalize features)      ║
║  3. Random Forest      (make prediction)         ║
╠══════════════════════════════════════════════════╣
║  INPUT FEATURES (8 total)                        ║
║  Pregnancies, Glucose, BloodPressure,            ║
║  SkinThickness, Insulin, BMI,                    ║
║  DiabetesPedigreeFunction, Age                   ║
╠══════════════════════════════════════════════════╣
║  OUTPUT                                          ║
║  0 = No Diabetes                                 ║
║  1 = Diabetes Detected                           ║
║  + Probability score for each class              ║
╚══════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════╗
║           MODEL CARD — Disease Predictor         ║
╠══════════════════════════════════════════════════╣
║  Model     : Random Forest Classifier            ║
║  Version   : 1.0.0                               ║
║  Task      : Binary Classification               ║
║  Dataset   : Pima Indians Diabetes (768 samples) ║
╠══════════════════════════════════════════════════╣
║  PERFORMANCE                                     ║
║  Recall    : 75.93%  (catches 76% of diabetics)  ║
║  ROC-AUC   : 83.00%  (strong separation ability) ║
║  Accuracy  : 77.27%                              ║
╠══════════════════════════════════════════════════╣
║  PIPELINE STEPS                                  ║
║  1. Median Imputation  (fix missing values)      ║
║  2. Standard Scaling   (normalize features)      ║
║  3. Random Forest      (make prediction)         ║
╠══════════════════════════════════════════════════╣
║  INPUT FEATURES (8 total)                  